
# V28 OFFICIAL — Bengali NLP Safety 0.600+ Reset

This notebook deliberately discards all earlier alternate-split predictions.

## Ground truth partition used here

- Train: **3,292**
- Validation: **500**
- Test: **1,000**
- `test.csv` IDs exactly equal `sample_submission.csv` IDs
- Train / validation / test IDs are disjoint

These are the current official competition files.

## Model

V28 uses four complementary signals:

1. **BanglaBERT** — Bengali-script specialist
2. **BanglishBERT** — Romanized/code-mixed Bengali specialist
3. **XLM-R base** — multilingual robustness
4. **Character TF-IDF logistic regression** — spelling noise, censorship and lexical toxicity

The three transformers use:

- 5-fold grouped OOF over **train + validation**
- official Bangla normalization where appropriate
- 3 epochs
- snapshot average of epochs 2 and 3
- asymmetric multi-label loss
- competition-shaped label weighting

The ensemble and thresholds are selected only from out-of-fold predictions.

**No old V14/V2 probabilities are used anywhere.**

At the end, send `v28_results.zip` to ChatGPT before Kaggle submission.


In [ ]:

# 1. Upload the OFFICIAL V28 data
from google.colab import files
import pathlib, zipfile

uploaded=files.upload()
assert "v28_OFFICIAL_data.zip" in uploaded, "Upload v28_OFFICIAL_data.zip"

DATA_DIR=pathlib.Path("/content/v28_data")
DATA_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile("v28_OFFICIAL_data.zip") as z:
    z.extractall(DATA_DIR)

print("Extracted:")
for p in sorted(DATA_DIR.iterdir()):
    print(" ",p.name,p.stat().st_size)


In [ ]:

# 2. Dependencies + model preflight
import sys, subprocess

subprocess.check_call([
    sys.executable,"-m","pip","install","-q",
    "transformers==4.57.6",
    "tokenizers==0.22.1",
    "sentencepiece==0.2.1",
    "scikit-learn","scipy","pandas","numpy","tqdm","safetensors"
])

subprocess.check_call([
    sys.executable,"-m","pip","install","-q",
    "git+https://github.com/csebuetnlp/normalizer"
])

from transformers import AutoConfig,AutoTokenizer

MODELS={
    "bangla":"csebuetnlp/banglabert",
    "banglish":"csebuetnlp/banglishbert",
    "xlmr":"FacebookAI/xlm-roberta-base",
}

for k,v in MODELS.items():
    print("Checking",k,v)
    _=AutoConfig.from_pretrained(v)
    _=AutoTokenizer.from_pretrained(v)

print("✅ All model checkpoints accessible.")


In [ ]:

# 3. Imports, OFFICIAL data verification, preprocessing
import gc,re,math,json,random,zipfile,unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset,DataLoader
from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from normalizer import normalize

SEED=2026
N_FOLDS=5
EPOCHS=3
SNAPSHOT_EPOCHS=[2,3]
MAX_LENGTH=128
TRAIN_BATCH=8
ACCUM=2
EVAL_BATCH=32
LR=1.5e-5
HEAD_LR=3e-5
WEIGHT_DECAY=0.01
WARMUP_RATIO=0.10

LABELS=[
    "is_toxic","implicit_toxicity","sarcasm","contains_profanity",
    "threatening","identity_attack","sexual_content"
]

MODELS={
    "bangla":"csebuetnlp/banglabert",
    "banglish":"csebuetnlp/banglishbert",
    "xlmr":"FacebookAI/xlm-roberta-base",
}

RUN_DIR=Path("/content/v28_output")
RUN_DIR.mkdir(exist_ok=True)

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(SEED)
assert torch.cuda.is_available(),"Enable T4 GPU"

train=pd.read_csv(DATA_DIR/"train.csv",dtype={"ID":str})
valid=pd.read_csv(DATA_DIR/"validation.csv",dtype={"ID":str})
test=pd.read_csv(DATA_DIR/"test.csv",dtype={"ID":str})
sample=pd.read_csv(DATA_DIR/"sample_submission.csv",dtype={"ID":str})

# STRICT OFFICIAL-SPLIT CHECKS
assert len(train)==3292, len(train)
assert len(valid)==500, len(valid)
assert len(test)==1000, len(test)
assert test["ID"].tolist()==sample["ID"].tolist()
assert set(train.ID).isdisjoint(set(valid.ID))
assert set(train.ID).isdisjoint(set(test.ID))
assert set(valid.ID).isdisjoint(set(test.ID))

print("✅ OFFICIAL partition verified: 3292 / 500 / 1000")
print("Train prevalence:")
display(train[LABELS].mean().round(4).to_frame("train"))
print("Validation prevalence:")
display(valid[LABELS].mean().round(4).to_frame("validation"))

def basic_clean(x):
    x=unicodedata.normalize("NFKC",str(x))
    x=x.replace("\\u200c"," ").replace("\\u200d"," ")
    return re.sub(r"\\s+"," ",x).strip()

def bn_clean(x):
    x=basic_clean(x)
    try:
        x=normalize(x)
    except Exception:
        pass
    return re.sub(r"\\s+"," ",x).strip()

def group_key(x):
    return basic_clean(x).lower()

def script_type(x):
    s=str(x)
    bn=bool(re.search(r"[\\u0980-\\u09FF]",s))
    lat=bool(re.search(r"[A-Za-z]",s))
    if bn and lat:
        return "mixed"
    if bn:
        return "native"
    return "roman"

all_labeled=pd.concat([train,valid],ignore_index=True)

for df in [all_labeled,test]:
    df["raw_text"]=df["text"].map(basic_clean)
    df["bn_text"]=df["text"].map(bn_clean)
    df["group_key"]=df["text"].map(group_key)
    df["script"]=df["text"].map(script_type)

print("Combined labeled:",len(all_labeled))
print("Script distributions:")
print("labeled:",all_labeled.script.value_counts(normalize=True).round(3).to_dict())
print("test:",test.script.value_counts(normalize=True).round(3).to_dict())


In [ ]:

# 4. Balanced grouped 5-fold split over ALL official labeled data
groups=all_labeled.groupby("group_key",sort=False).agg(
    size=("ID","size"),
    **{f"sum_{l}":(l,"sum") for l in LABELS}
).reset_index()

global_pos=np.array([all_labeled[l].sum() for l in LABELS],float)
total=float(len(all_labeled))
rarity=1/np.sqrt(np.maximum(global_pos,1))
mat=groups[[f"sum_{l}" for l in LABELS]].to_numpy(float)

rng=np.random.default_rng(SEED)
priority=(mat*rarity).sum(1)+0.005*groups["size"].to_numpy()
order=np.argsort(-(priority+rng.random(len(groups))*1e-8))

fp=np.zeros((N_FOLDS,len(LABELS)))
fs=np.zeros(N_FOLDS)
assign={}

for idx in order:
    row=groups.iloc[idx]
    vec=np.array([row[f"sum_{l}"] for l in LABELS],float)
    size=float(row["size"])

    costs=[]
    for f in range(N_FOLDS):
        p=fp.copy()
        s=fs.copy()
        p[f]+=vec
        s[f]+=size

        cost=(
            np.mean((p/np.maximum(global_pos[None,:],1)-1/N_FOLDS)**2)
            +0.30*np.mean((s/total-1/N_FOLDS)**2)
        )
        costs.append(cost)

    best=int(np.argmin(costs))
    assign[row["group_key"]]=best
    fp[best]+=vec
    fs[best]+=size

all_labeled["fold"]=all_labeled["group_key"].map(assign).astype(int)

assert all_labeled.groupby("group_key")["fold"].nunique().max()==1
assert set(all_labeled.fold)==set(range(N_FOLDS))

print("Fold sizes:",all_labeled.groupby("fold").size().to_dict())
display(all_labeled.groupby("fold")[LABELS].sum())


In [ ]:

# 5. Dataset + competition-shaped asymmetric loss
class DS(Dataset):
    def __init__(self,df,tok,text_col,labels=True):
        self.text=df[text_col].astype(str).tolist()
        self.y=df[LABELS].to_numpy(np.float32) if labels else None
        self.tok=tok

    def __len__(self):
        return len(self.text)

    def __getitem__(self,i):
        e=self.tok(
            self.text[i],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
        )
        if self.y is not None:
            e["labels"]=self.y[i]
        return e

class Coll:
    def __init__(self,tok):
        self.tok=tok

    def __call__(self,items):
        y=None
        if "labels" in items[0]:
            y=torch.tensor(
                np.stack([x.pop("labels") for x in items]),
                dtype=torch.float32
            )
        b=self.tok.pad(
            items,
            padding=True,
            pad_to_multiple_of=8,
            return_tensors="pt"
        )
        if y is not None:
            b["labels"]=y
        return b

METRIC_W=torch.tensor([0.5]+[0.5/6]*6,dtype=torch.float32)

def asl_loss(logits,y,gamma_neg=1.5,clip=0.05):
    p=torch.sigmoid(logits.float())
    pp=p
    pn=1-p

    if clip:
        pn=(pn+clip).clamp(max=1)

    loss=(
        y*torch.log(pp.clamp_min(1e-8))
        +(1-y)*torch.log(pn.clamp_min(1e-8))
    )

    pt=pp*y+pn*(1-y)
    gamma=gamma_neg*(1-y)
    loss*=torch.pow(1-pt,gamma)

    return (
        -loss*METRIC_W.to(logits.device).unsqueeze(0)
    ).sum(1).mean()

@torch.no_grad()
def predict(model,loader):
    model.eval()
    out=[]
    for b in loader:
        b.pop("labels",None)
        inp={k:v.to("cuda",non_blocking=True) for k,v in b.items()}
        with torch.autocast(device_type="cuda",dtype=torch.float16):
            logits=model(**inp).logits
        out.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(out)

def get_text_col(expert):
    return "bn_text" if expert in ["bangla","banglish"] else "raw_text"


In [ ]:

# 6. Train one transformer expert: 5-fold OOF + test ensemble
fold_seeds=[2026,17,3407,1337,42]

def train_expert(expert):
    name=MODELS[expert]
    text_col=get_text_col(expert)

    oof=np.zeros((len(all_labeled),len(LABELS)),np.float32)
    test_parts=[]
    hist=[]

    tok=AutoTokenizer.from_pretrained(name)

    for fold in range(N_FOLDS):
        tr=all_labeled[all_labeled.fold!=fold].copy()
        va=all_labeled[all_labeled.fold==fold].copy()

        assert set(tr.group_key).isdisjoint(set(va.group_key))

        cache=RUN_DIR/f"{expert}_fold_{fold}.npz"
        hp=RUN_DIR/f"{expert}_fold_{fold}_history.csv"

        if cache.exists():
            with np.load(cache) as z:
                idx=z["idx"]
                vp=z["valid"]
                tp=z["test"]
            if np.array_equal(idx,va.index.to_numpy()):
                print("✅ reuse",expert,"fold",fold)
                oof[idx]=vp
                test_parts.append(tp)
                continue

        print("\\n"+"="*70)
        print(expert,"fold",fold,"train",len(tr),"valid",len(va))

        seed_all(fold_seeds[fold])
        gc.collect()
        torch.cuda.empty_cache()

        c=Coll(tok)

        tl=DataLoader(
            DS(tr,tok,text_col,True),
            batch_size=TRAIN_BATCH,
            shuffle=True,
            num_workers=0,
            pin_memory=True,
            collate_fn=c,
        )
        vl=DataLoader(
            DS(va,tok,text_col,True),
            batch_size=EVAL_BATCH,
            shuffle=False,
            num_workers=0,
            pin_memory=True,
            collate_fn=c,
        )
        tel=DataLoader(
            DS(test,tok,text_col,False),
            batch_size=EVAL_BATCH,
            shuffle=False,
            num_workers=0,
            pin_memory=True,
            collate_fn=c,
        )

        model=AutoModelForSequenceClassification.from_pretrained(
            name,
            num_labels=len(LABELS),
            problem_type="multi_label_classification",
            ignore_mismatched_sizes=True,
        ).to("cuda")

        no_decay=["bias","LayerNorm.weight","layer_norm.weight"]
        groups=[
            {
                "params":[p for n,p in model.named_parameters()
                          if p.requires_grad and not any(x in n for x in no_decay)
                          and ("classifier" not in n and "score" not in n)],
                "lr":LR,
                "weight_decay":WEIGHT_DECAY,
            },
            {
                "params":[p for n,p in model.named_parameters()
                          if p.requires_grad and any(x in n for x in no_decay)
                          and ("classifier" not in n and "score" not in n)],
                "lr":LR,
                "weight_decay":0.0,
            },
            {
                "params":[p for n,p in model.named_parameters()
                          if p.requires_grad and ("classifier" in n or "score" in n)],
                "lr":HEAD_LR,
                "weight_decay":WEIGHT_DECAY,
            },
        ]
        groups=[g for g in groups if len(g["params"])>0]

        opt=torch.optim.AdamW(groups)

        steps=math.ceil(len(tl)/ACCUM)*EPOCHS
        sched=get_linear_schedule_with_warmup(
            opt,
            int(steps*WARMUP_RATIO),
            steps
        )
        scaler=torch.amp.GradScaler("cuda")

        sv=[]
        st=[]
        fh=[]

        for epoch in range(1,EPOCHS+1):
            model.train()
            opt.zero_grad(set_to_none=True)
            run=0.0
            seen=0

            for step,b in enumerate(
                tqdm(tl,desc=f"{expert} f{fold} e{epoch}/{EPOCHS}")
            ):
                y=b.pop("labels").to("cuda",non_blocking=True)
                inp={k:v.to("cuda",non_blocking=True) for k,v in b.items()}

                with torch.autocast(device_type="cuda",dtype=torch.float16):
                    logits=model(**inp).logits

                loss=asl_loss(logits,y)
                scaler.scale(loss/ACCUM).backward()

                run+=float(loss.item())*len(y)
                seen+=len(y)

                if (step+1)%ACCUM==0 or step+1==len(tl):
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)

                    old=scaler.get_scale()
                    scaler.step(opt)
                    scaler.update()

                    if scaler.get_scale()>=old:
                        sched.step()

                    opt.zero_grad(set_to_none=True)

            rec={"expert":expert,"fold":fold,"epoch":epoch,"loss":run/max(seen,1)}
            print(rec)
            fh.append(rec)
            hist.append(rec)
            pd.DataFrame(fh).to_csv(hp,index=False)

            if epoch in SNAPSHOT_EPOCHS:
                sv.append(predict(model,vl))
                st.append(predict(model,tel))

        vp=np.mean(sv,axis=0)
        tp=np.mean(st,axis=0)

        oof[va.index.to_numpy()]=vp
        test_parts.append(tp)

        np.savez_compressed(
            cache,
            idx=va.index.to_numpy(),
            valid=vp,
            test=tp,
        )

        del model,opt,sched,scaler,tl,vl,tel
        gc.collect()
        torch.cuda.empty_cache()

    test_prob=np.mean(test_parts,axis=0)

    np.savez_compressed(
        RUN_DIR/f"{expert}_oof_test.npz",
        oof=oof,
        test=test_prob
    )
    pd.DataFrame(hist).to_csv(
        RUN_DIR/f"{expert}_training_history.csv",
        index=False
    )

    return oof,test_prob


In [ ]:

# 7. Train the three transformer experts
outputs={}

for expert in ["bangla","banglish","xlmr"]:
    print("\\n"+"#"*80)
    print("TRAINING",expert.upper())
    print("#"*80)
    outputs[expert]=train_expert(expert)

bangla_oof,bangla_test=outputs["bangla"]
banglish_oof,banglish_test=outputs["banglish"]
xlmr_oof,xlmr_test=outputs["xlmr"]

print("✅ Transformer training complete.")


In [ ]:

# 8. Character TF-IDF 5-fold OOF expert
char_oof=np.zeros((len(all_labeled),len(LABELS)),np.float32)
char_test_parts=[]

for fold in range(N_FOLDS):
    tr=all_labeled[all_labeled.fold!=fold]
    va=all_labeled[all_labeled.fold==fold]

    vec=TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(2,6),
        min_df=2,
        max_features=180000,
        sublinear_tf=True,
        dtype=np.float32,
    )

    Xtr=vec.fit_transform(tr.raw_text)
    Xva=vec.transform(va.raw_text)
    Xte=vec.transform(test.raw_text)

    pv=np.zeros((len(va),len(LABELS)),np.float32)
    pt=np.zeros((len(test),len(LABELS)),np.float32)

    for j,l in enumerate(LABELS):
        clf=LogisticRegression(
            C=3.0,
            solver="liblinear",
            max_iter=1000,
            random_state=SEED+fold,
        )
        clf.fit(Xtr,tr[l].to_numpy(int))
        pv[:,j]=clf.predict_proba(Xva)[:,1]
        pt[:,j]=clf.predict_proba(Xte)[:,1]

    char_oof[va.index.to_numpy()]=pv
    char_test_parts.append(pt)

    print("char fold",fold,"features",Xtr.shape[1])

char_test=np.mean(char_test_parts,axis=0)

np.savez_compressed(
    RUN_DIR/"char_oof_test.npz",
    oof=char_oof,
    test=char_test,
)

print("✅ Character expert complete.")


In [ ]:

# 9. Fast exact competition metric + nested threshold calibration
y=all_labeled[LABELS].to_numpy(np.int8)
folds=all_labeled.fold.to_numpy(int)

def f1_pos(y,p):
    y=np.asarray(y,np.int8); p=np.asarray(p,np.int8)
    tp=np.sum((y==1)&(p==1))
    fp=np.sum((y==0)&(p==1))
    fn=np.sum((y==1)&(p==0))
    d=2*tp+fp+fn
    return float(2*tp/d) if d else 0.0

def f1_macro(y,p):
    vals=[]
    for c in [0,1]:
        tp=np.sum((y==c)&(p==c))
        fp=np.sum((y!=c)&(p==c))
        fn=np.sum((y==c)&(p!=c))
        d=2*tp+fp+fn
        vals.append(2*tp/d if d else 0.0)
    return float(np.mean(vals))

def score_label(label,y,p):
    return f1_macro(y,p) if label=="is_toxic" else f1_pos(y,p)

def best_threshold(label,y,p):
    vals=np.unique(p)
    if len(vals)>700:
        vals=np.quantile(vals,np.linspace(0,1,700))
    cand=np.unique(np.r_[0.0,vals,1.000001])

    best=-1
    best_t=0.5
    for t in cand:
        s=score_label(label,y,(p>=t).astype(np.int8))
        if s>best:
            best=s
            best_t=float(t)
    return best_t

def crossfit(label,y,p):
    pred=np.zeros(len(y),np.int8)
    fs=[]
    ts=[]

    for f in range(N_FOLDS):
        held=folds==f
        t=best_threshold(label,y[~held],p[~held])
        pred[held]=(p[held]>=t).astype(np.int8)
        ts.append(t)
        fs.append(score_label(label,y[held],pred[held]))

    return {
        "score":score_label(label,y,pred),
        "fold_scores":np.asarray(fs,float),
        "thresholds":np.asarray(ts,float),
        "prediction":pred,
    }

def logit(p):
    p=np.clip(np.asarray(p,float),1e-6,1-1e-6)
    return np.log(p/(1-p))

def sigmoid(z):
    return 1/(1+np.exp(-z))


In [ ]:

# 10. Robust per-label ensemble search
# Candidate families are intentionally constrained to reduce meta-overfitting.

script=all_labeled.script.to_numpy(str)
test_script=test.script.to_numpy(str)

def make_candidate(kind,j,is_test=False,char_w=0.0,mode="prob"):
    if is_test:
        b=bangla_test[:,j]
        g=banglish_test[:,j]
        x=xlmr_test[:,j]
        c=char_test[:,j]
        s=test_script
    else:
        b=bangla_oof[:,j]
        g=banglish_oof[:,j]
        x=xlmr_oof[:,j]
        c=char_oof[:,j]
        s=script

    if kind=="avg3":
        tr=(b+g+x)/3
    elif kind=="bangla_xlmr":
        tr=.6*b+.4*x
    elif kind=="banglish_xlmr":
        tr=.6*g+.4*x
    elif kind=="script_gate":
        native=s=="native"
        tr=np.empty(len(b),float)
        tr[native]=.6*b[native]+.4*x[native]
        tr[~native]=.6*g[~native]+.4*x[~native]
    elif kind=="xlmr":
        tr=x.copy()
    elif kind=="bangla":
        tr=b.copy()
    elif kind=="banglish":
        tr=g.copy()
    else:
        raise ValueError(kind)

    if char_w<=0:
        return tr

    if mode=="prob":
        return (1-char_w)*tr+char_w*c

    return sigmoid(
        (1-char_w)*logit(tr)
        +char_w*logit(c)
    )

kinds=[
    "avg3","bangla_xlmr","banglish_xlmr","script_gate",
    "xlmr","bangla","banglish"
]
char_ws=[0.0,0.10,0.20,0.30,0.40]

rows=[]
best_by_label={}

for j,label in enumerate(LABELS):
    yy=y[:,j]
    truth_rate=float(yy.mean())
    best=None

    for kind in kinds:
        for cw in char_ws:
            modes=["prob"] if cw==0 else ["prob","logit"]

            for mode in modes:
                p=make_candidate(kind,j,False,cw,mode)
                cv=crossfit(label,yy,p)

                mean=float(cv["fold_scores"].mean())
                std=float(cv["fold_scores"].std())
                pred_rate=float(cv["prediction"].mean())

                # Mild prevalence/stability penalty for rare-label reliability.
                prevalence_gap=abs(pred_rate-truth_rate)
                robust=mean-0.05*std-0.03*prevalence_gap

                rec={
                    "label":label,
                    "kind":kind,
                    "char_weight":cw,
                    "mode":mode,
                    "crossfit_score":float(cv["score"]),
                    "mean_fold":mean,
                    "std_fold":std,
                    "truth_rate":truth_rate,
                    "pred_rate":pred_rate,
                    "robust_score":robust,
                    "median_threshold":float(np.median(cv["thresholds"])),
                    "fold_scores":"|".join(f"{z:.6f}" for z in cv["fold_scores"]),
                }
                rows.append(rec)

                key=(robust,cv["score"],-std,-cw)
                if best is None or key>best["key"]:
                    best={
                        "key":key,
                        "rec":rec,
                        "cv":cv,
                    }

    best_by_label[label]=best

search=pd.DataFrame(rows)
search.to_csv(RUN_DIR/"v28_ensemble_search.csv",index=False)

locked=pd.DataFrame([best_by_label[l]["rec"] for l in LABELS])
locked.to_csv(RUN_DIR/"v28_locked_config.csv",index=False)

print("LOCKED V28 CONFIG")
display(locked.round(5))


In [ ]:

# 11. Compute OOF official score and create OFFICIAL test submission
oof_pred=np.zeros((len(all_labeled),len(LABELS)),np.int8)
test_pred=np.zeros((len(test),len(LABELS)),np.int8)

per_label=[]

for j,label in enumerate(LABELS):
    b=best_by_label[label]
    rec=b["rec"]

    p_oof=make_candidate(
        rec["kind"],j,False,
        rec["char_weight"],rec["mode"]
    )
    p_test=make_candidate(
        rec["kind"],j,True,
        rec["char_weight"],rec["mode"]
    )

    # OOF prediction is the nested-threshold prediction used for honest reporting.
    oof_pred[:,j]=b["cv"]["prediction"]

    # Test uses median of thresholds learned without each held fold.
    t=float(rec["median_threshold"])
    test_pred[:,j]=(p_test>=t).astype(np.int8)

    per_label.append(
        score_label(label,y[:,j],oof_pred[:,j])
    )

tox=float(per_label[0])
attrs=float(np.mean(per_label[1:]))
official=float(.5*tox+.5*attrs)

report=pd.DataFrame({
    "label":LABELS,
    "oof_score":per_label,
    "true_rate":y.mean(0),
    "oof_pred_rate":oof_pred.mean(0),
    "test_pred_rate":test_pred.mean(0),
})
report.to_csv(RUN_DIR/"v28_oof_report.csv",index=False)

submission=sample.copy()
for j,l in enumerate(LABELS):
    submission[l]=test_pred[:,j]

assert submission.ID.tolist()==test.ID.tolist()
assert submission[LABELS].isin([0,1]).all().all()

submission.to_csv(
    RUN_DIR/"submission_v28_OFFICIAL.csv",
    index=False
)

summary={
    "version":"v28_OFFICIAL_correct_split",
    "train_rows":3292,
    "validation_rows":500,
    "combined_labeled_rows":3792,
    "test_rows":1000,
    "test_ids_match_sample":True,
    "oof_toxic_macro_f1":tox,
    "oof_attribute_mean_f1":attrs,
    "oof_official_score":official,
    "target":0.600,
    "SUBMIT_CANDIDATE":bool(official>=0.600),
    "IMPORTANT":"This is the first reset model using only the current official 3292/500/1000 split."
}
(RUN_DIR/"v28_summary.json").write_text(json.dumps(summary,indent=2))

print(json.dumps(summary,indent=2))
display(report.round(5))

if official>=0.600:
    print("\\n✅ OFFLINE OOF TARGET PASSED. Send results ZIP to ChatGPT before submission.")
else:
    print("\\n⚠️ OOF below 0.600. Still send results ZIP; do not submit yet.")


In [ ]:

# 12. Package V28 results
np.savez_compressed(
    RUN_DIR/"v28_probabilities.npz",
    bangla_oof=bangla_oof,
    bangla_test=bangla_test,
    banglish_oof=banglish_oof,
    banglish_test=banglish_test,
    xlmr_oof=xlmr_oof,
    xlmr_test=xlmr_test,
    char_oof=char_oof,
    char_test=char_test,
)

zip_path=Path("/content/v28_results.zip")
with zipfile.ZipFile(zip_path,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for p in RUN_DIR.iterdir():
        if p.is_file():
            z.write(p,arcname=p.name)

print("Created:",zip_path)

from google.colab import files
files.download(str(zip_path))
